# Train the Q50 paper-style critic

Primary Q-Planning analogue. Each offline action input is **50 actions actually executed across five replanning intervals**; its reward target spans the same 50 environment steps and bootstraps at the saved state 50 steps later. At deployment, a generated 50-action candidate would be scored but only its first 10 actions executed. Uncertainty is not used in this training run.

Run once with `RUN_MODE = 'smoke'`. If all ten updates and both validation passes finish, change only that line to `'full'` for the fixed 8,000-update run.

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SNAPSHOT_ID = 'PASTE_PCPCDS_SNAPSHOT_ID'  # immutable snapshot from notebook 56
RUN_MODE = 'smoke'  # change to 'full' only after the smoke run succeeds
MICRO_BATCH_SIZE = 8  # effective batch remains 64 via accumulation
CACHE_DOWNLOAD_WORKERS = 4  # bounded parallel Supabase artifact downloads
STREAM_WINDOWS = True  # avoid a second Q50 copy of the large source cache
CACHE_ROOT = '/content/qplanning_cache'  # fast local runtime disk
OUTPUT_ROOT = '/content/drive/MyDrive/pnp_qplanning_corrector'
assert RUN_MODE in ('smoke', 'full')
print({'critic': 'Q50-paper-style', 'run_mode': RUN_MODE,
       'micro_batch_size': MICRO_BATCH_SIZE,
       'cache_download_workers': CACHE_DOWNLOAD_WORKERS,
       'stream_windows': STREAM_WINDOWS, 'snapshot_id': SNAPSHOT_ID})

In [ ]:
from pnp.qplanning_critic import run_qplanning_training_test

report = run_qplanning_training_test(
    snapshot_id=SNAPSHOT_ID, horizon=50, run_mode=RUN_MODE,
    cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT,
    micro_batch_size=MICRO_BATCH_SIZE,
    cache_download_workers=CACHE_DOWNLOAD_WORKERS,
    stream_windows=STREAM_WINDOWS, resume=True)

In [ ]:
import pandas as pd
display(pd.DataFrame([{
    'critic': f"Q{report['horizon']}",
    'mode': report['run_mode'],
    'train_windows': report['train_windows'],
    'validation_windows': report['validation_windows'],
    **report['validation'],
}]))
print('checkpoint:', report['final_checkpoint'])